# Bootstrapping beluga detections from model output 

As of spring of 2022, the beluga detector trained by Microsoft has some trouble distinguishing anthropogenic noise from true beluga. A manual review of output from deployment 237 revealed numerous instances where non-beluga sounds were identified as beluga with above-expected confidience. 

We intend to reuse this dataset as training data for the model. The overall objective is to improve the model's ability to differentiate between beluga and noise.

Given a table of false positive beluga generated by beluga detecting AI and the true value of the detection, we will convert the output back into the format of the input. Put differently, we will set the "beluga or noise" value as the species ID and will revert the column format back to what is expected by the spectrogram extraction process

In [120]:
import pandas as pd
import sqlite3 as sq3
import os
from datetime import datetime

## Data from 237

This data is the one that has high scores of Beluga, but are actually noise. 

The spectrogram filenames are based on the number of seconds into the wave file. We need to extract this and add it to the UTC detection time. This is because as of May 2022, the ML output does not include seconds in the detection field.



In [194]:
 
dataframe = pd.read_excel(r"D:\beluga\Data\Special_Data\raw\237D_ML-PG_WandM_Detector_output.xlsx")
fp_237 = dataframe[(dataframe['AB Validation']=="F")] # filter to just the manually verified detections

#extract the seconds from the spectrogram file name
fp_237['wave_detection_seconds'] = fp_237["spectrogram_filename"].apply(lambda x:os.path.basename(x).split("_")[-1].split(".")[0] )
fp_237['wave_detection_seconds'] = fp_237['wave_detection_seconds'].astype(int) # was a string, now a number


"""
extract hour minute and second of wave file. Wave files are named by detector ID.YYMMDDHHMMSS
e.g. 67158025.190907001502.wav. The last 6 numbers is the time the wave file started: 00:15:02
"""
fp_237['wave_start_time'] = fp_237["spectrogram_filename"].apply(lambda x:datetime.strptime("20" + x.split(".")[1].split("_")[0], "%Y%m%d%H%M%S"))

#combine the wave start time with the spectrogram start time to get the false detection time
fp_237['new_detection_time'] = fp_237['wave_start_time'] + pd.to_timedelta(fp_237['wave_detection_seconds'], unit="s")

fp_237_training = fp_237.drop(labels=["UTC",
                                      'Detection_TimeStamp',
                                      'Date',
                                      "audio_filename", 
                                      "spectrogram_filename", 
                                      "predicted_probability",
                                      "predicted_detection","wave_detection_seconds","wave_start_time"],axis=1)
fp_237_training.rename(columns={"AB Validation":"Species", "new_detection_time":"UTC"})

fp_237_training

C:\Users\mml\.conda\envs\beluga\lib\site-packages\ipykernel_launcher.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """
C:\Users\mml\.conda\envs\beluga\lib\site-packages\ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
C:\Users\mml\.conda\envs\beluga\lib\site-packages\ipykernel_launcher.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: ht

,Id,UID,AB Validation,AB comments,new_detection_time
400,116535,12590000004,F,mechanical noise,2019-09-07 00:18:11
401,116536,12590000005,F,mechanical noise,2019-09-07 00:18:11
402,116537,12590000006,F,mechanical noise,2019-09-07 00:18:11
403,116538,12590000007,F,mechanical noise,2019-09-07 00:18:11
404,116539,12590000008,F,mechanical noise,2019-09-07 00:18:11
...,...,...,...,...,...
119975,61583,7936000004,F,unknown,2019-07-20 12:45:20
119980,61772,7943000015,F,unknown,2019-07-20 14:33:25
119981,61773,7943000016,F,unknown,2019-07-20 14:33:25
119982,61774,7943000017,F,unknown,2019-07-20 14:33:25


In [197]:
fp_237_training.rename(columns={"AB Validation":"Species", "new_detection_time":"UTC"}, inplace=True)
fp_237_training.to_csv("fp_237_training_data_bootstrap.csv")

## This dataset is 216 and is the validation of model ouput.

In [156]:
fp_216 = pd.read_excel(r"D:\beluga\Data\Special_Data\raw\ML_Results_False_Positives_AB validation (1).xlsx")

fp_216['Manually_Validated_Call'].replace({"N":"F","Y":"B"}, inplace=True)

fp_216['wave_start_time'] = fp_216["spectrogram_filename"].apply(lambda x:datetime.strptime("20" + os.path.basename(x).split(".")[1].split("_")[0], "%Y%m%d%H%M%S"))

fp_216['new_detection_time'] = fp_216['wave_start_time'] + pd.to_timedelta(fp_216['Start_Time'], unit="s")

,spectrogram_filename,audio_filename,spectrogram_start_second,predicted_probability,wav_file,date,Start_Time,End_Time,spectrogram_start_timestamp,spectrogram_end_timestamp,Manually_Validated_Call,AB validation notes
0,E:/Whale_Acoustics/Data/Extracted_Spectrogram_...,67158025.18,160,0.984307,180821000000,180821,160,162,180821000000,180821000000,N,Mine ends at 158.9 and ML picks up some very b...
1,E:/Whale_Acoustics/Data/Extracted_Spectrogram_...,67158025.18,236,0.979034,180821000000,180821,236,238,180821000000,180821000000,N,"Same as above, ML picked up squeek only a few ..."
2,E:/Whale_Acoustics/Data/Extracted_Spectrogram_...,67158025.18,96,0.968933,180829000000,180829,96,98,180829000000,180829000000,N,True-beluga. ML picked up very faint calls inb...
3,E:/Whale_Acoustics/Data/Extracted_Spectrogram_...,67158025.18,24,0.961895,180829000000,180829,24,26,180829000000,180829000000,N,True-beluga. ML picked up very faint calls inb...
4,E:/Whale_Acoustics/Data/Extracted_Spectrogram_...,67158025.18,240,0.948792,180821000000,180821,240,242,180821000000,180821000000,N,ML picked up squeek only a few ms long that I ...


In [192]:
fp_216_training = fp_216.drop(labels=["spectrogram_filename",
                                      "audio_filename", 
                                      "spectrogram_start_second", 
                                      "predicted_probability",
                                      "wav_file",
                                      "date",
                                      "Start_Time",
                                      "End_Time", 
                                      "spectrogram_start_timestamp", 
                                      "spectrogram_end_timestamp", 
                                      "wave_start_time"], axis=1)

fp_216_training.rename(columns={"new_detection_time":"UTC", "Manually_Validated_Call": "Species"}, inplace=True)
fp_216_training.to_csv("fp_216_training_data_bootstrap.csv")

,Manually_Validated_Call,AB validation notes,new_detection_time
0,F,Mine ends at 158.9 and ML picks up some very b...,2018-08-21 22:47:42
1,F,"Same as above, ML picked up squeek only a few ...",2018-08-21 22:48:58
2,F,True-beluga. ML picked up very faint calls inb...,2018-08-29 16:16:38
3,F,True-beluga. ML picked up very faint calls inb...,2018-08-29 16:15:26
4,F,ML picked up squeek only a few ms long that I ...,2018-08-21 22:49:02
...,...,...,...
215995,F,NaN,2018-09-05 15:17:08
215996,F,NaN,2018-09-01 08:47:46
215997,F,NaN,2018-09-03 06:19:52
215998,F,NaN,2018-08-16 08:18:26
